# 04_Clean_Document_Text

Tareas:

1. No modifica las tablas raw de extracción;
2. Procesa únicamente documentos con `cleaning_status` pendiente o fallido;
3. Usa únicamente las secciones de la `extraction_run_id` vigente de cada documento;
4. Aplica una limpieza ligera y conservadora para texto científico;
5. Preserva estructura, referencias y trazabilidad;
6. Evita que secciones vacías entren en la reconstrucción del documento;
7. Reemplaza de forma autoritativa las secciones limpias de cada documento procesado, evitando filas obsoletas de ejecuciones anteriores;
8. No crea chunks: esa responsabilidad permanece en el script 05.

In [0]:
from __future__ import annotations

import json
import re
import unicodedata
import uuid

from datetime import datetime, timezone
from typing import Any

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# ============================================================
# Configuración
# ============================================================

CATALOG_NAME = "workspace"
SCHEMA_NAME = "tfm_pmc"

INVENTORY_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_inventory"
SOURCE_DOCUMENT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_extracted_documents"
SOURCE_SECTION_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_extracted_sections"

CLEAN_SECTION_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_clean_sections"
CLEAN_DOCUMENT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_clean_documents"
PIPELINE_RUNS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pipeline_runs"

PIPELINE_NAME = "04_Clean_Document_Text_v3_Clean"
CLEANING_VERSION = "scientific_section_clean_v3"

MAX_DOCUMENTS = 20

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

print(f"Run ID: {RUN_ID}")
print(f"Raw sections: {SOURCE_SECTION_TABLE}")
print(f"Clean sections: {CLEAN_SECTION_TABLE}")
print(f"Clean documents: {CLEAN_DOCUMENT_TABLE}")


Run ID: 460d5cd5-95d3-495d-ba67-21bdceb792c2
Raw sections: workspace.tfm_pmc.pmc_extracted_sections
Clean sections: workspace.tfm_pmc.pmc_clean_sections
Clean documents: workspace.tfm_pmc.pmc_clean_documents


In [0]:
# ============================================================
# Crear tablas persistentes
# ============================================================

spark.sql(
    f'''
    CREATE TABLE IF NOT EXISTS {CLEAN_SECTION_TABLE} (
        pmcid STRING NOT NULL,
        article_version STRING NOT NULL,

        section_index INT NOT NULL,
        section_id STRING,
        parent_section_id STRING,

        section_type STRING,
        section_title STRING,
        section_path STRING,
        section_level INT,

        raw_character_count BIGINT,
        raw_word_count BIGINT,

        clean_content STRING,

        clean_character_count BIGINT,
        clean_word_count BIGINT,

        is_reference_section BOOLEAN,
        include_in_retrieval BOOLEAN,

        extraction_source STRING,

        cleaning_status STRING,
        cleaning_version STRING,

        extraction_run_id STRING,
        cleaning_run_id STRING,

        processed_at TIMESTAMP,
        updated_at TIMESTAMP,

        error_message STRING
    )
    USING DELTA
    '''
)

spark.sql(
    f'''
    CREATE TABLE IF NOT EXISTS {CLEAN_DOCUMENT_TABLE} (
        pmcid STRING NOT NULL,
        article_version STRING NOT NULL,

        title STRING,
        journal STRING,
        publication_date STRING,
        publication_year INT,
        author_names ARRAY<STRING>,

        clean_text STRING,

        section_count INT,
        retrievable_section_count INT,
        reference_section_count INT,

        clean_character_count BIGINT,
        clean_word_count BIGINT,

        cleaning_status STRING,
        cleaning_version STRING,

        cleaning_run_id STRING,

        processed_at TIMESTAMP,
        updated_at TIMESTAMP,

        error_message STRING
    )
    USING DELTA
    '''
)

print("Clean tables are ready.")

Clean tables are ready.


In [0]:
# ============================================================
# Funciones de limpieza
# ============================================================

CONTROL_CHARACTERS_RE = re.compile(
    r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]"
)

MULTIPLE_SPACES_RE = re.compile(
    r"[ \t]+"
)

MULTIPLE_NEWLINES_RE = re.compile(
    r"\n{3,}"
)

SPACE_BEFORE_PUNCT_RE = re.compile(
    r"\s+([,.;:!?])"
)

# Conservador: no elimina símbolos científicos, nombres de genes,
# variantes, unidades ni puntuación biomédica.
def clean_scientific_text(
    text: str | None,
) -> str:

    if not text:
        return ""

    value = unicodedata.normalize(
        "NFKC",
        text,
    )

    value = CONTROL_CHARACTERS_RE.sub(
        " ",
        value,
    )

    value = value.replace(
        "\r\n",
        "\n",
    ).replace(
        "\r",
        "\n",
    )

    value = MULTIPLE_SPACES_RE.sub(
        " ",
        value,
    )

    value = MULTIPLE_NEWLINES_RE.sub(
        "\n\n",
        value,
    )

    value = SPACE_BEFORE_PUNCT_RE.sub(
        r"\1",
        value,
    )

    return value.strip()

In [0]:
# ============================================================
# Seleccionar documentos pendientes
# ============================================================

source_documents_df = (
    spark.table(SOURCE_DOCUMENT_TABLE)
    .filter(F.col("extraction_successful") == True)
    .select(
        "pmcid",
        "article_version",
        "title",
        "extraction_run_id",
    )
)

inventory_metadata_df = (
    spark.table(INVENTORY_TABLE)
    .filter(F.col("is_latest_version") == True)
    .select(
        "pmcid",
        "article_version",
        "title",
        "journal",
        "publication_date",
        "publication_year",
        "author_names",
        "cleaning_status",
    )
)

pending_documents_df = (
    source_documents_df.alias("source")
    .join(
        inventory_metadata_df.alias("inventory"),
        on=["pmcid", "article_version"],
        how="inner",
    )
    .filter(
        F.coalesce(
            F.col("inventory.cleaning_status"),
            F.lit("pending"),
        ).isin("pending", "failed")
    )
    .select(
        "pmcid",
        "article_version",
        F.coalesce(
            F.col("inventory.title"),
            F.col("source.title"),
        ).alias("title"),
        "journal",
        "publication_date",
        "publication_year",
        "author_names",
        F.col("source.extraction_run_id").alias("extraction_run_id"),
    )
    .orderBy("pmcid", "article_version")
    .limit(MAX_DOCUMENTS)
)

documents_selected = pending_documents_df.count()

print("Documents selected:", documents_selected)

if documents_selected > 0:
    display(pending_documents_df)
else:
    print("No pending documents to clean.")


Documents selected: 20


pmcid,article_version,title,journal,publication_date,publication_year,author_names,extraction_run_id
PMC13538227,PMC13538227.1,Hereditary connective tissue disorders in unselected patients with spontaneous cervical artery dissection: a targeted next generation sequencing approach and systematic review,Neurological sciences : official journal of the Italian Neurological Society and of the Italian Society of Clinical Neurophysiology,2026 Sep 2,2026,"List(Corradi L, Ferraro C, Tesi F, Abrignani G, Castellini P, Latte L, Trapasso MC, Genovese A, Ritelli MG, Cinquina V, Giliani SC, Magoni M, Menozzi R, Pezzini A)",ee5d4e24-a318-4df9-a751-5738bd2ea162
PMC13538279,PMC13538279.1,Validation of a Cellular Imaging‐Based Method as a Potential Biomarker for SPG4 Hereditary Spastic Paraplegia,Annals of clinical and translational neurology,2026 Sep 2,2026,"List(Fattorini G, Licursi V, Zanna GD, Dal Canto F, Barghigiani M, Setola N, Rossi S, Funcis A, Santorelli FM, Silvestri G, Casali C, Sardina F, Rinaldo C)",ee5d4e24-a318-4df9-a751-5738bd2ea162
PMC13539412,PMC13539412.1,Parathyroid carcinoma: epidemiology and genetics,"Endocrine oncology (Bristol, England)",2026 Jan,2026,"List(Betea D, Petrossians P)",ee5d4e24-a318-4df9-a751-5738bd2ea162
PMC13540131,PMC13540131.2,Development of an electrochemiluminescence-based bridging assay to detect antibodies against a PTH inverse agonist in human plasma,Bioanalysis,2026 Jul,2026,"List(Nduwumwami AJ, Wagner EJ, Wang AQ, Fang Y, Tao D, Xu X)",ee5d4e24-a318-4df9-a751-5738bd2ea162
PMC13543812,PMC13543812.1,"Equity in genome sequencing for rare disease diagnosis: a cross-sectional analysis of data from the UK 100,000 Genomes Project",EBioMedicine,2026 Sep,2026,"List(Tallman S, Moutsianas L, Nguyen T, Cho Y, Mackintosh M, Kasperaviciute D, Brown MA, Ellingford JM, Kuchenbaecker K, Silver MJ)",ee5d4e24-a318-4df9-a751-5738bd2ea162
PMC13544149,PMC13544149.1,Identification of a Novel CDH2 Gene Variant in an ACOGS Patient with Concurrent Respiratory Tract Infection: A Case Report,"Pediatric health, medicine and therapeutics",2026,2026,"List(Lu Y, Fang F, Zhou H, Shu S, Liu X)",ee5d4e24-a318-4df9-a751-5738bd2ea162
PMC13545448,PMC13545448.1,Artificial intelligence-assisted clinical exome sequencing: Insights and outcomes from 822 pediatric diagnoses,Genetics in medicine open,2026,2026,"List(Pan Y, Danley P, Kramer T, Noruzinia M, Buser K, Yatsenko AN, Bellissimo D, Guo F)",ee5d4e24-a318-4df9-a751-5738bd2ea162
PMC13547081,PMC13547081.1,Unmasking Mucopolysaccharidosis Type I in a Patient With Wolf–Hirschhorn Syndrome: Diagnostic Overshadowing,JIMD reports,2026 Sep,2026,"List(Cifuentes-Uribe K, Girard S, Froissart R, Pettazzoni M, Guffon N)",ee5d4e24-a318-4df9-a751-5738bd2ea162
PMC13547469,PMC13547469.1,Case Report: Successful hematopoietic stem cell transplantation in pediatric pyruvate kinase deficiency: a single-center Asian case series demonstrating favorable outcomes,Frontiers in immunology,2026,2026,"List(Yan H, Li D, Lu X, Yang X, Zhu Y, Sun S)",ee5d4e24-a318-4df9-a751-5738bd2ea162
PMC13552502,PMC13552502.1,The socioeconomic impacts of fibrodysplasia ossificans progressiva: evidence from a retrospective case-control study in France,JBMR plus,2026 Oct,2026,"List(Baujat G, Bouée S, Solanke O, Croskery K, Kirion J, Cormier-Daire V, Jannot AS, Jeanbat V, Funck-Brentano T)",ee5d4e24-a318-4df9-a751-5738bd2ea162


In [0]:
# ============================================================
# Obtener únicamente las secciones de la extracción vigente
# ============================================================

selected_keys_df = (
    pending_documents_df
    .select(
        "pmcid",
        "article_version",
        "extraction_run_id",
    )
)

sections_to_clean_df = (
    spark.table(SOURCE_SECTION_TABLE).alias("sections")
    .join(
        selected_keys_df.alias("selected"),
        on=[
            "pmcid",
            "article_version",
            "extraction_run_id",
        ],
        how="inner",
    )
    .select(
        "pmcid",
        "article_version",
        "section_index",
        "section_id",
        "parent_section_id",
        "section_type",
        "section_title",
        "section_path",
        "section_level",
        "content",
        "is_reference_section",
        "include_in_retrieval",
        "extraction_source",
        "extraction_run_id",
    )
    .orderBy(
        "pmcid",
        "article_version",
        "section_index",
    )
)

sections_selected = sections_to_clean_df.count()

print("Sections selected:", sections_selected)

if sections_selected > 0:
    display(sections_to_clean_df)


Sections selected: 382


pmcid article_version section_index section_id parent_section_id section_type section_title section_path section_level content is_reference_section include_in_retrieval extraction_source extraction_run_id PMC13538227 PMC13538227.1 0 abstract null abstract Abstract Abstract 0 Introduction Whether spontaneous cervical artery dissection (sCeAD), the leading cause of ischemic stroke in young adults, represents the manifestation of unrecognized hereditary connective tissue disorders (HCTDs) and whether HCTDs have a major impact in the epidemiology of the disease is a matter of ongoing debate. We aimed at determining the frequency of clinically relevant genetic variants (CRGVs) in a cohort of unselected sCeAD patients by targeted next-generation sequencing (NGS) approach. Methods We designed a high-throughput sequencing panel to identify variants in 38 candidate genes associated with arterial dissection or aneurysm and screened patients with apparently sporadic sCeAD, consecutively referred to one comprehensive stroke center from August 2020 to December 2025. The frequency of known disease-causing and pertinent variants of uncertain significance (VUS) was calculated. Then, we performed a systematic review of all studies evaluating the prevalence of monogenic disorders among sCeAD patients up to December 2025. Results Among 183 patients (males, 51.3%; mean age, 42.0 ± 11.3 years), 2 (1.1%) carried a CeAD-causing variant in COL3A1 ( NM_000090.4 :c.2959G > A:p.Gly987Ser) and ABCC6 ( NM_001351800.1 :c.3071G > A:p.Arg1024Gln), respectively. In addition, we identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of CeAD. Conclusion Systematic search for rare disease-causing variants should not be recommended in all sCeAD cases but it should be limited to selected individuals with a high pre-test probability to harbor a monogenic disease. Supplementary Information The online version contains supplementary material available at https://doi.org/10.1007/s10072-026-09308-6 . false true jats_xml ee5d4e24-a318-4df9-a751-5738bd2ea162 PMC13538227 PMC13538227.1 1 Sec1 null section Introduction Introduction 1 Arterial dissection is, by definition, the accumulation of blood within the wall of an artery. Once considered uncommon, dissections of carotid or vertebral arteries (cervical artery dissections, CeADs) are now recognized as the major cause of ischemic stroke in young adults, accounting for approximately 20% of cases in patients under 45 years of age. Notwithstanding, the pathogenesis of CeAD is still poorly defined, especially in those cases that occur spontaneously (spontaneous CeAD, sCeAD), without any identifiable precipitating events [ 1 ]. Several arguments point toward a key role played by the connective tissue component of the arterial wall. The finding of composite collagen fibrils and fragmented elastic fibers on electron microscopic examination of skin biopsy specimens in more than half of patients with sCeAD [ 2 – 4 ] and the identification of clinically detectable signs of connective tissue aberration in most sCeAD patients support the hypothesis of a generalized structural connective tissue defect predisposing to disease occurrence, even in sporadic cases without evident signs of a known hereditary connective tissue disorder (HCTD) [ 5 , 6 ]. In this view, the reduced or altered biosynthesis of the extracellular matrix (ECM) components might lead to a generalized structural weakness of the vessel wall and explain the increased risk for sCeAD. Whether such arteriopathy underlying sCeAD represents a mild phenotypic manifestation of HCTD has never been fully elucidated. Most of the studies so far have been conducted on small series with limited number of screened genes [ 7 , 8 ]. Recently, next-generation sequencing (NGS) has become an e

In [0]:
# ============================================================
# Procesar secciones
# ============================================================

section_results: list[dict[str, Any]] = []

for index, record in enumerate(
    sections_to_clean_df.toLocalIterator(),
    start=1,
):
    processed_at = datetime.now(
        timezone.utc
    )

    raw_text = (
        record["content"]
        or ""
    )

    raw_character_count = len(
        raw_text
    )

    raw_word_count = len(
        raw_text.split()
    )

    try:
        clean_text = clean_scientific_text(
            raw_text
        )

        clean_character_count = len(
            clean_text
        )

        clean_word_count = len(
            clean_text.split()
        )

        cleaning_status = (
            "completed"
            if clean_text
            else "empty"
        )

        error_message = None

    except Exception as error:
        clean_text = ""
        clean_character_count = 0
        clean_word_count = 0
        cleaning_status = "failed"
        error_message = str(error)

    section_results.append({
        "pmcid": record["pmcid"],
        "article_version": record["article_version"],

        "section_index": record["section_index"],
        "section_id": record["section_id"],
        "parent_section_id": record["parent_section_id"],

        "section_type": record["section_type"],
        "section_title": record["section_title"],
        "section_path": record["section_path"],
        "section_level": record["section_level"],

        "raw_character_count": raw_character_count,
        "raw_word_count": raw_word_count,

        "clean_content": clean_text,

        "clean_character_count": clean_character_count,
        "clean_word_count": clean_word_count,

        "is_reference_section": record[
            "is_reference_section"
        ],

        "include_in_retrieval": record[
            "include_in_retrieval"
        ],

        "extraction_source": record[
            "extraction_source"
        ],

        "cleaning_status": cleaning_status,
        "cleaning_version": CLEANING_VERSION,

        "extraction_run_id": record[
            "extraction_run_id"
        ],
        "cleaning_run_id": RUN_ID,

        "processed_at": processed_at,
        "updated_at": processed_at,

        "error_message": error_message,
    })

print(
    "Sections processed:",
    len(section_results),
)

Sections processed: 382


In [0]:
# ============================================================
# Crear DataFrame de secciones limpias
# ============================================================

clean_sections_df = None

if section_results:
    clean_sections_df = spark.createDataFrame(
        section_results,
        schema=spark.table(CLEAN_SECTION_TABLE).schema,
    )

    print("Clean sections DataFrame ready:", clean_sections_df.count())


Clean sections DataFrame ready: 382


In [0]:
# ============================================================
# Reemplazo autoritativo de secciones limpias
# ============================================================

# Para cada documento procesado eliminamos primero cualquier versión
# anterior de sus secciones limpias. Así no quedan filas obsoletas si
# una nueva extracción produce menos secciones o cambia sus índices.

if documents_selected > 0:
    clean_sections_delta = DeltaTable.forName(
        spark,
        CLEAN_SECTION_TABLE,
    )

    processed_document_keys_df = (
        pending_documents_df
        .select("pmcid", "article_version")
        .distinct()
    )

    (
        clean_sections_delta.alias("target")
        .merge(
            processed_document_keys_df.alias("source"),
            '''
            target.pmcid = source.pmcid
            AND target.article_version = source.article_version
            ''',
        )
        .whenMatchedDelete()
        .execute()
    )

    if clean_sections_df is not None:
        (
            clean_sections_df
            .write
            .mode("append")
            .saveAsTable(CLEAN_SECTION_TABLE)
        )

    print("Clean sections replaced for processed documents.")


Clean sections replaced for processed documents.


In [0]:
# ============================================================
# Reconstruir documentos desde secciones limpias
# ============================================================

clean_document_df = None

if clean_sections_df is not None:

    usable_sections_df = (
        clean_sections_df
        .filter(
            (F.col("cleaning_status") == "completed")
            & (F.length(F.trim(F.col("clean_content"))) > 0)
        )
    )

    document_text_df = (
        usable_sections_df
        .groupBy("pmcid", "article_version")
        .agg(
            F.concat_ws(
                "\n\n",
                F.transform(
                    F.array_sort(
                        F.collect_list(
                            F.struct(
                                F.col("section_index"),
                                F.col("section_title"),
                                F.col("clean_content"),
                            )
                        )
                    ),
                    lambda x: F.concat(
                        F.lit("## "),
                        F.coalesce(
                            x["section_title"],
                            F.lit("Untitled section"),
                        ),
                        F.lit("\n"),
                        x["clean_content"],
                    ),
                ),
            ).alias("clean_text")
        )
    )

    document_metrics_df = (
        clean_sections_df
        .groupBy("pmcid", "article_version")
        .agg(
            F.count("*").alias("section_count"),

            F.sum(
                F.when(
                    (F.col("include_in_retrieval") == True)
                    & (F.col("cleaning_status") == "completed")
                    & (F.length(F.trim(F.col("clean_content"))) > 0),
                    1,
                ).otherwise(0)
            ).alias("retrievable_section_count"),

            F.sum(
                F.col("is_reference_section").cast("int")
            ).alias("reference_section_count"),

            F.sum("clean_character_count").alias(
                "clean_character_count"
            ),

            F.sum("clean_word_count").alias(
                "clean_word_count"
            ),

            F.sum(
                F.when(
                    F.col("cleaning_status") == "failed",
                    1,
                ).otherwise(0)
            ).alias("failed_sections"),
        )
    )

    clean_document_df = (
        document_metrics_df
        .join(
            document_text_df,
            on=["pmcid", "article_version"],
            how="left",
        )
        .join(
            pending_documents_df.drop("extraction_run_id"),
            on=["pmcid", "article_version"],
            how="inner",
        )
        .withColumn(
            "clean_text",
            F.coalesce(F.col("clean_text"), F.lit("")),
        )
        .withColumn(
            "cleaning_status",
            F.when(
                F.col("failed_sections") > 0,
                F.lit("completed_with_errors"),
            ).when(
                F.col("retrievable_section_count") == 0,
                F.lit("failed"),
            ).otherwise(
                F.lit("completed")
            ),
        )
        .withColumn(
            "cleaning_version",
            F.lit(CLEANING_VERSION),
        )
        .withColumn(
            "cleaning_run_id",
            F.lit(RUN_ID),
        )
        .withColumn(
            "processed_at",
            F.current_timestamp(),
        )
        .withColumn(
            "updated_at",
            F.current_timestamp(),
        )
        .withColumn(
            "error_message",
            F.when(
                F.col("retrievable_section_count") == 0,
                F.lit("No usable retrievable sections after cleaning."),
            ).when(
                F.col("failed_sections") > 0,
                F.concat(
                    F.lit("Sections failed: "),
                    F.col("failed_sections").cast("string"),
                ),
            ).otherwise(
                F.lit(None).cast("string")
            ),
        )
        .select(
            "pmcid",
            "article_version",
            "title",
            "journal",
            "publication_date",
            "publication_year",
            "author_names",
            "clean_text",
            "section_count",
            "retrievable_section_count",
            "reference_section_count",
            "clean_character_count",
            "clean_word_count",
            "cleaning_status",
            "cleaning_version",
            "cleaning_run_id",
            "processed_at",
            "updated_at",
            "error_message",
        )
    )

    display(clean_document_df)


pmcid article_version title journal publication_date publication_year author_names clean_text section_count retrievable_section_count reference_section_count clean_character_count clean_word_count cleaning_status cleaning_version cleaning_run_id processed_at updated_at error_message PMC13538227 PMC13538227.1 Hereditary connective tissue disorders in unselected patients with spontaneous cervical artery dissection: a targeted next generation sequencing approach and systematic review Neurological sciences : official journal of the Italian Neurological Society and of the Italian Society of Clinical Neurophysiology 2026 Sep 2 2026 List(Corradi L, Ferraro C, Tesi F, Abrignani G, Castellini P, Latte L, Trapasso MC, Genovese A, Ritelli MG, Cinquina V, Giliani SC, Magoni M, Menozzi R, Pezzini A) ## Abstract
Introduction Whether spontaneous cervical artery dissection (sCeAD), the leading cause of ischemic stroke in young adults, represents the manifestation of unrecognized hereditary connective tissue disorders (HCTDs) and whether HCTDs have a major impact in the epidemiology of the disease is a matter of ongoing debate. We aimed at determining the frequency of clinically relevant genetic variants (CRGVs) in a cohort of unselected sCeAD patients by targeted next-generation sequencing (NGS) approach. Methods We designed a high-throughput sequencing panel to identify variants in 38 candidate genes associated with arterial dissection or aneurysm and screened patients with apparently sporadic sCeAD, consecutively referred to one comprehensive stroke center from August 2020 to December 2025. The frequency of known disease-causing and pertinent variants of uncertain significance (VUS) was calculated. Then, we performed a systematic review of all studies evaluating the prevalence of monogenic disorders among sCeAD patients up to December 2025. Results Among 183 patients (males, 51.3%; mean age, 42.0 ± 11.3 years), 2 (1.1%) carried a CeAD-causing variant in COL3A1 ( NM_000090.4:c.2959G > A:p.Gly987Ser) and ABCC6 ( NM_001351800.1:c.3071G > A:p.Arg1024Gln), respectively. In addition, we identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of CeAD. Conclusion Systematic search for rare disease-causing variants should not be recommended in all sCeAD cases but it should be limited to selected individuals with a high pre-test probability to harbor a monogenic disease. Supplementary Information The online version contains supplementary material available at https://doi.org/10.1007/s10072-026-09308-6.

## Introduction
Arterial dissection is, by definition, the accumulation of blood within the wall of an artery. Once considered uncommon, dissections of carotid or vertebral arteries (cervical artery dissections, CeADs) are now recognized as the major cause of ischemic stroke in young adults, accounting for approximately 20% of cases in patients under 45 years of age. Notwithstanding, the pathogenesis of CeAD is still poorly defined, especially in those cases that occur spontaneously (spontaneous CeAD, sCeAD), without any identifiable precipitating events [ 1 ]. Several arguments point toward a key role played by the connective tissue component of the arterial wall. The finding of composite collagen fibrils and fragmented elastic fibers on electron microscopic examination of skin biopsy specimens in more than half of patients with sCeAD [ 2 – 4 ] and the identification of clinically detectable signs of connective tissue aberration in most sCeAD patients support the hypothesis of a generalized structural connective tissue defect predisposing to disease occurrence, even in sporadic cases without evident signs of a known hereditary connective tissue disorder (HCTD) [ 5, 6 ]. In this view, the reduced or altered biosynthesis of the extracellular matrix (ECM) c

In [0]:
# ============================================================
# MERGE documentos limpios
# ============================================================

if clean_document_df is not None:
    clean_documents_delta = DeltaTable.forName(
        spark,
        CLEAN_DOCUMENT_TABLE,
    )

    (
        clean_documents_delta.alias("target")
        .merge(
            clean_document_df.alias("source"),
            '''
            target.pmcid = source.pmcid
            AND target.article_version = source.article_version
            ''',
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        "Clean documents MERGE completed."
    )

Clean documents MERGE completed.


In [0]:
# ============================================================
# Actualizar cleaning_status en pmc_inventory
# ============================================================

# IMPORTANTE PARA SERVERLESS:
# No reutilizamos clean_document_df después del MERGE porque Spark podría
# reevaluar su lineage tras modificar pmc_inventory.
# En su lugar, leemos los registros ya persistidos de esta ejecución.

persisted_clean_documents_df = (
    spark.table(
        CLEAN_DOCUMENT_TABLE
    )
    .filter(
        F.col(
            "cleaning_run_id"
        ) == RUN_ID
    )
)

inventory_delta = DeltaTable.forName(
    spark,
    INVENTORY_TABLE,
)

status_df = (
    persisted_clean_documents_df
    .select(
        "pmcid",
        "article_version",

        F.when(
            F.col(
                "cleaning_status"
            ).isin(
                "completed",
                "completed_with_errors",
            ),
            F.lit("completed"),
        )
        .otherwise(
            F.lit("failed")
        )
        .alias(
            "new_cleaning_status"
        ),

        "error_message",
        "updated_at",
    )
)

(
    inventory_delta.alias("target")
    .merge(
        status_df.alias("source"),
        '''
        target.pmcid = source.pmcid
        AND target.article_version = source.article_version
        ''',
    )
    .whenMatchedUpdate(
        set={
            "cleaning_status": (
                "source.new_cleaning_status"
            ),
            "error_message": (
                "source.error_message"
            ),
            "updated_at": (
                "source.updated_at"
            ),
        }
    )
    .execute()
)

print(
    "pmc_inventory cleaning statuses updated."
)


pmc_inventory cleaning statuses updated.


In [0]:
# ============================================================
# Métricas de calidad
# ============================================================

# Se leen desde Delta usando cleaning_run_id para evitar cualquier
# reevaluación lazy del DataFrame original.

metrics_documents_df = (
    spark.table(
        CLEAN_DOCUMENT_TABLE
    )
    .filter(
        F.col(
            "cleaning_run_id"
        ) == RUN_ID
    )
)

metrics_sections_df = (
    spark.table(
        CLEAN_SECTION_TABLE
    )
    .filter(
        F.col(
            "cleaning_run_id"
        ) == RUN_ID
    )
)

cleaning_metrics_df = (
    metrics_documents_df
    .agg(
        F.count("*").alias(
            "documents_processed"
        ),

        F.sum(
            "section_count"
        ).alias(
            "sections_processed"
        ),

        F.sum(
            "retrievable_section_count"
        ).alias(
            "retrievable_sections"
        ),

        F.sum(
            "reference_section_count"
        ).alias(
            "reference_sections"
        ),

        F.sum(
            "clean_character_count"
        ).alias(
            "clean_characters"
        ),

        F.sum(
            "clean_word_count"
        ).alias(
            "clean_words"
        ),

        F.sum(
            F.when(
                F.col(
                    "publication_year"
                ).isNotNull(),
                1,
            ).otherwise(0)
        ).alias(
            "documents_with_year"
        ),

        F.sum(
            F.when(
                F.size(
                    F.col(
                        "author_names"
                    )
                ) > 0,
                1,
            ).otherwise(0)
        ).alias(
            "documents_with_authors"
        ),
    )
)

display(
    cleaning_metrics_df
)

display(
    metrics_sections_df
    .groupBy(
        "is_reference_section",
        "include_in_retrieval",
    )
    .count()
)


documents_processed,sections_processed,retrievable_sections,reference_sections,clean_characters,clean_words,documents_with_year,documents_with_authors
20,382,342,20,906048,130626,20,20


is_reference_section,include_in_retrieval,count
false,true,362
true,false,20


In [0]:
# ============================================================
# Registrar ejecución
# ============================================================

RUN_COMPLETED_AT = datetime.now(
    timezone.utc
)

records_processed = (
    metrics_documents_df.count()
)

records_failed = (
    metrics_documents_df
    .filter(
        ~F.col(
            "cleaning_status"
        ).isin(
            "completed",
            "completed_with_errors",
        )
    )
    .count()
)

records_successful = (
    records_processed
    - records_failed
)

run_row = spark.createDataFrame(
    [
        (
            RUN_ID,
            PIPELINE_NAME,
            (
                "completed"
                if records_failed == 0
                else "completed_with_errors"
            ),
            RUN_STARTED_AT,
            RUN_COMPLETED_AT,
            int(MAX_DOCUMENTS),
            int(documents_selected),
            int(records_processed),
            int(records_successful),
            0,
            int(records_failed),
            json.dumps({
                "source_table": SOURCE_SECTION_TABLE,
                "clean_section_table": CLEAN_SECTION_TABLE,
                "clean_document_table": CLEAN_DOCUMENT_TABLE,
                "cleaning_version": CLEANING_VERSION,
                "references_preserved": True,
                "references_excluded_from_retrieval": True,
                "serverless_safe": True,
            }),
            None,
        )
    ],
    schema=T.StructType([
        T.StructField("run_id", T.StringType(), False),
        T.StructField("pipeline_name", T.StringType(), False),
        T.StructField("run_status", T.StringType(), False),
        T.StructField("started_at", T.TimestampType(), False),
        T.StructField("completed_at", T.TimestampType(), True),
        T.StructField("records_requested", T.LongType(), True),
        T.StructField("records_found", T.LongType(), True),
        T.StructField("records_processed", T.LongType(), True),
        T.StructField("records_inserted", T.LongType(), True),
        T.StructField("records_updated", T.LongType(), True),
        T.StructField("records_failed", T.LongType(), True),
        T.StructField("execution_metadata", T.StringType(), True),
        T.StructField("error_message", T.StringType(), True),
    ]),
)

run_row.write.mode(
    "append"
).saveAsTable(
    PIPELINE_RUNS_TABLE
)

print(
    f"Pipeline run registered: {RUN_ID}"
)


Pipeline run registered: 460d5cd5-95d3-495d-ba67-21bdceb792c2


In [0]:
# ============================================================
# Auditoría final de la ejecución
# ============================================================

print("===== INVENTORY =====")
display(
    spark.table(INVENTORY_TABLE)
    .filter(F.col("is_latest_version") == True)
    .select(
        "pmcid",
        "article_version",
        "extraction_status",
        "cleaning_status",
        "error_message",
    )
    .orderBy("pmcid", "article_version")
)

print("===== CLEAN DOCUMENTS - CURRENT RUN =====")
display(
    spark.table(CLEAN_DOCUMENT_TABLE)
    .filter(F.col("cleaning_run_id") == RUN_ID)
    .orderBy("pmcid", "article_version")
)

print("===== CLEAN SECTIONS - CURRENT RUN =====")
display(
    spark.table(CLEAN_SECTION_TABLE)
    .filter(F.col("cleaning_run_id") == RUN_ID)
    .orderBy("pmcid", "article_version", "section_index")
)

print("===== PIPELINE RUN =====")
display(
    spark.table(PIPELINE_RUNS_TABLE)
    .filter(F.col("run_id") == RUN_ID)
)


===== INVENTORY =====


pmcid,article_version,extraction_status,cleaning_status,error_message
PMC13538227,PMC13538227.1,completed,completed,null
PMC13538279,PMC13538279.1,completed,completed,null
PMC13539412,PMC13539412.1,completed,completed,null
PMC13540131,PMC13540131.2,completed,completed,null
PMC13543812,PMC13543812.1,completed,completed,null
PMC13544149,PMC13544149.1,completed,completed,null
PMC13545448,PMC13545448.1,completed,completed,null
PMC13547081,PMC13547081.1,completed,completed,null
PMC13547469,PMC13547469.1,completed,completed,null
PMC13552502,PMC13552502.1,completed,completed,null


===== CLEAN DOCUMENTS - CURRENT RUN =====


pmcid article_version title journal publication_date publication_year author_names clean_text section_count retrievable_section_count reference_section_count clean_character_count clean_word_count cleaning_status cleaning_version cleaning_run_id processed_at updated_at error_message PMC13538227 PMC13538227.1 Hereditary connective tissue disorders in unselected patients with spontaneous cervical artery dissection: a targeted next generation sequencing approach and systematic review Neurological sciences : official journal of the Italian Neurological Society and of the Italian Society of Clinical Neurophysiology 2026 Sep 2 2026 List(Corradi L, Ferraro C, Tesi F, Abrignani G, Castellini P, Latte L, Trapasso MC, Genovese A, Ritelli MG, Cinquina V, Giliani SC, Magoni M, Menozzi R, Pezzini A) ## Abstract
Introduction Whether spontaneous cervical artery dissection (sCeAD), the leading cause of ischemic stroke in young adults, represents the manifestation of unrecognized hereditary connective tissue disorders (HCTDs) and whether HCTDs have a major impact in the epidemiology of the disease is a matter of ongoing debate. We aimed at determining the frequency of clinically relevant genetic variants (CRGVs) in a cohort of unselected sCeAD patients by targeted next-generation sequencing (NGS) approach. Methods We designed a high-throughput sequencing panel to identify variants in 38 candidate genes associated with arterial dissection or aneurysm and screened patients with apparently sporadic sCeAD, consecutively referred to one comprehensive stroke center from August 2020 to December 2025. The frequency of known disease-causing and pertinent variants of uncertain significance (VUS) was calculated. Then, we performed a systematic review of all studies evaluating the prevalence of monogenic disorders among sCeAD patients up to December 2025. Results Among 183 patients (males, 51.3%; mean age, 42.0 ± 11.3 years), 2 (1.1%) carried a CeAD-causing variant in COL3A1 ( NM_000090.4:c.2959G > A:p.Gly987Ser) and ABCC6 ( NM_001351800.1:c.3071G > A:p.Arg1024Gln), respectively. In addition, we identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of CeAD. Conclusion Systematic search for rare disease-causing variants should not be recommended in all sCeAD cases but it should be limited to selected individuals with a high pre-test probability to harbor a monogenic disease. Supplementary Information The online version contains supplementary material available at https://doi.org/10.1007/s10072-026-09308-6.

## Introduction
Arterial dissection is, by definition, the accumulation of blood within the wall of an artery. Once considered uncommon, dissections of carotid or vertebral arteries (cervical artery dissections, CeADs) are now recognized as the major cause of ischemic stroke in young adults, accounting for approximately 20% of cases in patients under 45 years of age. Notwithstanding, the pathogenesis of CeAD is still poorly defined, especially in those cases that occur spontaneously (spontaneous CeAD, sCeAD), without any identifiable precipitating events [ 1 ]. Several arguments point toward a key role played by the connective tissue component of the arterial wall. The finding of composite collagen fibrils and fragmented elastic fibers on electron microscopic examination of skin biopsy specimens in more than half of patients with sCeAD [ 2 – 4 ] and the identification of clinically detectable signs of connective tissue aberration in most sCeAD patients support the hypothesis of a generalized structural connective tissue defect predisposing to disease occurrence, even in sporadic cases without evident signs of a known hereditary connective tissue disorder (HCTD) [ 5, 6 ]. In this view, the reduced or altered biosynthesis of the extracellular matrix (ECM) c

===== CLEAN SECTIONS - CURRENT RUN =====


pmcid article_version section_index section_id parent_section_id section_type section_title section_path section_level raw_character_count raw_word_count clean_content clean_character_count clean_word_count is_reference_section include_in_retrieval extraction_source cleaning_status cleaning_version extraction_run_id cleaning_run_id processed_at updated_at error_message PMC13538227 PMC13538227.1 0 abstract null abstract Abstract Abstract 0 1916 268 Introduction Whether spontaneous cervical artery dissection (sCeAD), the leading cause of ischemic stroke in young adults, represents the manifestation of unrecognized hereditary connective tissue disorders (HCTDs) and whether HCTDs have a major impact in the epidemiology of the disease is a matter of ongoing debate. We aimed at determining the frequency of clinically relevant genetic variants (CRGVs) in a cohort of unselected sCeAD patients by targeted next-generation sequencing (NGS) approach. Methods We designed a high-throughput sequencing panel to identify variants in 38 candidate genes associated with arterial dissection or aneurysm and screened patients with apparently sporadic sCeAD, consecutively referred to one comprehensive stroke center from August 2020 to December 2025. The frequency of known disease-causing and pertinent variants of uncertain significance (VUS) was calculated. Then, we performed a systematic review of all studies evaluating the prevalence of monogenic disorders among sCeAD patients up to December 2025. Results Among 183 patients (males, 51.3%; mean age, 42.0 ± 11.3 years), 2 (1.1%) carried a CeAD-causing variant in COL3A1 ( NM_000090.4:c.2959G > A:p.Gly987Ser) and ABCC6 ( NM_001351800.1:c.3071G > A:p.Arg1024Gln), respectively. In addition, we identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of CeAD. Conclusion Systematic search for rare disease-causing variants should not be recommended in all sCeAD cases but it should be limited to selected individuals with a high pre-test probability to harbor a monogenic disease. Supplementary Information The online version contains supplementary material available at https://doi.org/10.1007/s10072-026-09308-6. 1913 265 false true jats_xml completed scientific_section_clean_v3 ee5d4e24-a318-4df9-a751-5738bd2ea162 460d5cd5-95d3-495d-ba67-21bdceb792c2 2026-09-13T22:02:04.857Z 2026-09-13T22:02:04.857Z null PMC13538227 PMC13538227.1 1 Sec1 null section Introduction Introduction 1 2277 350 Arterial dissection is, by definition, the accumulation of blood within the wall of an artery. Once considered uncommon, dissections of carotid or vertebral arteries (cervical artery dissections, CeADs) are now recognized as the major cause of ischemic stroke in young adults, accounting for approximately 20% of cases in patients under 45 years of age. Notwithstanding, the pathogenesis of CeAD is still poorly defined, especially in those cases that occur spontaneously (spontaneous CeAD, sCeAD), without any identifiable precipitating events [ 1 ]. Several arguments point toward a key role played by the connective tissue component of the arterial wall. The finding of composite collagen fibrils and fragmented elastic fibers on electron microscopic examination of skin biopsy specimens in more than half of patients with sCeAD [ 2 – 4 ] and the identification of clinically detectable signs of connective tissue aberration in most sCeAD patients support the hypothesis of a generalized structural connective tissue defect predisposing to disease occurrence, even in sporadic cases without evident signs of a known hereditary connective tissue disorder (HCTD) [ 5, 6 ]. In this view, the reduced or altered biosynthesis of the extracellular matrix (ECM) components might lead to a generalized structural weakness of the vessel wall and explain the inc

===== PIPELINE RUN =====


run_id,pipeline_name,run_status,started_at,completed_at,records_requested,records_found,records_processed,records_inserted,records_updated,records_failed,execution_metadata,error_message
460d5cd5-95d3-495d-ba67-21bdceb792c2,04_Clean_Document_Text_v3_Clean,completed,2026-09-13T22:01:44.458Z,2026-09-13T22:02:31.159Z,20,20,20,20,0,0,"{""source_table"": ""workspace.tfm_pmc.pmc_extracted_sections"", ""clean_section_table"": ""workspace.tfm_pmc.pmc_clean_sections"", ""clean_document_table"": ""workspace.tfm_pmc.pmc_clean_documents"", ""cleaning_version"": ""scientific_section_clean_v3"", ""references_preserved"": true, ""references_excluded_from_retrieval"": true, ""serverless_safe"": true}",null
